# End-to-End Project: Image Classification Application

A computer vision project with CNN architecture and inference pipeline.

## Project Overview

**Objective**: Build a robust image classifier with data augmentation and model deployment.

**Skills Applied**:
- CNN architecture design
- Data augmentation strategies
- Transfer learning concepts
- Inference pipeline creation
- Model evaluation and visualization

In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
import json

# Sklearn imports
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings('ignore')
np.random.seed(42)

print("Libraries loaded successfully!")

## 1. Data Generation

We'll create synthetic image data representing different object categories.

In [ ]:
def generate_synthetic_images(n_samples_per_class=500, img_size=32, n_classes=5):
    """
    Generate synthetic image data with distinct patterns for each class.
    
    Classes:
    0 - Circles (dots in center)
    1 - Horizontal lines
    2 - Vertical lines
    3 - Diagonal lines
    4 - Grid patterns
    """
    np.random.seed(42)
    
    class_names = ['circle', 'horizontal', 'vertical', 'diagonal', 'grid']
    
    images = []
    labels = []
    
    for class_idx in range(n_classes):
        for _ in range(n_samples_per_class):
            # Create base image with noise
            img = np.random.normal(0.2, 0.05, (img_size, img_size, 3))
            
            # Add class-specific patterns
            if class_idx == 0:  # Circle
                center = img_size // 2 + np.random.randint(-3, 4)
                radius = np.random.randint(5, 10)
                y, x = np.ogrid[:img_size, :img_size]
                mask = (x - center)**2 + (y - center)**2 <= radius**2
                color = np.random.uniform(0.6, 1.0, 3)
                img[mask] = color
                
            elif class_idx == 1:  # Horizontal lines
                n_lines = np.random.randint(2, 5)
                for _ in range(n_lines):
                    y = np.random.randint(3, img_size - 3)
                    thickness = np.random.randint(1, 3)
                    color = np.random.uniform(0.6, 1.0, 3)
                    img[y:y+thickness, :, :] = color
                    
            elif class_idx == 2:  # Vertical lines
                n_lines = np.random.randint(2, 5)
                for _ in range(n_lines):
                    x = np.random.randint(3, img_size - 3)
                    thickness = np.random.randint(1, 3)
                    color = np.random.uniform(0.6, 1.0, 3)
                    img[:, x:x+thickness, :] = color
                    
            elif class_idx == 3:  # Diagonal
                color = np.random.uniform(0.6, 1.0, 3)
                offset = np.random.randint(-5, 5)
                for i in range(img_size):
                    j = i + offset
                    if 0 <= j < img_size:
                        img[i, j, :] = color
                        if i+1 < img_size:
                            img[i+1, j, :] = color
                            
            elif class_idx == 4:  # Grid
                spacing = np.random.randint(4, 8)
                color = np.random.uniform(0.6, 1.0, 3)
                for i in range(0, img_size, spacing):
                    img[i, :, :] = color
                    img[:, i, :] = color
            
            # Clip values
            img = np.clip(img, 0, 1)
            images.append(img)
            labels.append(class_idx)
    
    return np.array(images), np.array(labels), class_names


# Generate dataset
X, y, class_names = generate_synthetic_images(500, img_size=32, n_classes=5)

print(f"Dataset shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Classes: {class_names}")
print(f"Samples per class: {np.bincount(y)}")

In [ ]:
# Visualize sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for class_idx in range(5):
    class_images = X[y == class_idx]
    
    # First row: first sample
    axes[0, class_idx].imshow(class_images[0])
    axes[0, class_idx].set_title(f'{class_names[class_idx].title()}')
    axes[0, class_idx].axis('off')
    
    # Second row: random sample
    axes[1, class_idx].imshow(class_images[np.random.randint(len(class_images))])
    axes[1, class_idx].axis('off')

plt.suptitle('Sample Images from Each Class', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Data Preprocessing and Splitting

In [ ]:
# Split data: 70% train, 15% validation, 15% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp  # 0.176 of 85% = 15%
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## 3. Data Augmentation

Implementing custom augmentation functions for image data.

In [ ]:
class ImageAugmenter:
    """
    Image augmentation class with various transformations.
    """
    
    def __init__(self, rotation_range=15, horizontal_flip=True, 
                 brightness_range=(0.8, 1.2), noise_std=0.05):
        self.rotation_range = rotation_range
        self.horizontal_flip = horizontal_flip
        self.brightness_range = brightness_range
        self.noise_std = noise_std
    
    def rotate(self, image, angle):
        """Simple rotation by shifting pixels (approximate)."""
        if angle == 0:
            return image
        # For simplicity, use np.rot90 for 90-degree rotations
        k = int(angle / 90) % 4
        return np.rot90(image, k)
    
    def flip_horizontal(self, image):
        """Flip image horizontally."""
        return np.fliplr(image)
    
    def flip_vertical(self, image):
        """Flip image vertically."""
        return np.flipud(image)
    
    def adjust_brightness(self, image, factor):
        """Adjust image brightness."""
        return np.clip(image * factor, 0, 1)
    
    def add_noise(self, image):
        """Add Gaussian noise."""
        noise = np.random.normal(0, self.noise_std, image.shape)
        return np.clip(image + noise, 0, 1)
    
    def augment(self, image):
        """Apply random augmentations."""
        aug_image = image.copy()
        
        # Random horizontal flip
        if self.horizontal_flip and np.random.random() > 0.5:
            aug_image = self.flip_horizontal(aug_image)
        
        # Random brightness
        brightness_factor = np.random.uniform(*self.brightness_range)
        aug_image = self.adjust_brightness(aug_image, brightness_factor)
        
        # Random noise
        if np.random.random() > 0.5:
            aug_image = self.add_noise(aug_image)
        
        return aug_image
    
    def generate_augmented_batch(self, images, labels, augment_factor=2):
        """Generate augmented dataset."""
        aug_images = [images]
        aug_labels = [labels]
        
        for _ in range(augment_factor - 1):
            batch_aug = np.array([self.augment(img) for img in images])
            aug_images.append(batch_aug)
            aug_labels.append(labels)
        
        return np.vstack(aug_images), np.hstack(aug_labels)


# Create augmenter
augmenter = ImageAugmenter()

# Demonstrate augmentation
sample_image = X_train[0]

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes[0, 0].imshow(sample_image)
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

for i in range(1, 10):
    row, col = i // 5, i % 5
    aug_img = augmenter.augment(sample_image)
    axes[row, col].imshow(aug_img)
    axes[row, col].set_title(f'Augmented {i}')
    axes[row, col].axis('off')

plt.suptitle('Data Augmentation Examples', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Generate augmented training data
X_train_aug, y_train_aug = augmenter.generate_augmented_batch(
    X_train, y_train, augment_factor=3
)

# Shuffle
shuffle_idx = np.random.permutation(len(y_train_aug))
X_train_aug = X_train_aug[shuffle_idx]
y_train_aug = y_train_aug[shuffle_idx]

print(f"Original training: {X_train.shape[0]} samples")
print(f"Augmented training: {X_train_aug.shape[0]} samples")
print(f"Augmentation factor: {X_train_aug.shape[0] / X_train.shape[0]:.1f}x")

## 4. CNN Model Architecture

Building a simple CNN using only NumPy (educational purpose).

In [ ]:
class SimpleCNN:
    """
    Simple CNN implementation for educational purposes.
    Uses flattened features + fully connected layers.
    """
    
    def __init__(self, input_shape, n_classes, hidden_units=128):
        self.input_shape = input_shape
        self.n_classes = n_classes
        self.hidden_units = hidden_units
        
        # Calculate input size (flattened)
        self.input_size = np.prod(input_shape)
        
        # Initialize weights
        self._initialize_weights()
        
    def _initialize_weights(self):
        """Xavier initialization for weights."""
        scale1 = np.sqrt(2.0 / self.input_size)
        scale2 = np.sqrt(2.0 / self.hidden_units)
        
        self.W1 = np.random.randn(self.input_size, self.hidden_units) * scale1
        self.b1 = np.zeros(self.hidden_units)
        
        self.W2 = np.random.randn(self.hidden_units, self.hidden_units // 2) * scale2
        self.b2 = np.zeros(self.hidden_units // 2)
        
        self.W3 = np.random.randn(self.hidden_units // 2, self.n_classes) * scale2
        self.b3 = np.zeros(self.n_classes)
    
    def relu(self, x):
        return np.maximum(0, x)
    
    def relu_derivative(self, x):
        return (x > 0).astype(float)
    
    def softmax(self, x):
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)
    
    def forward(self, X):
        """Forward pass."""
        # Flatten input
        self.X_flat = X.reshape(X.shape[0], -1)
        
        # Layer 1
        self.z1 = self.X_flat @ self.W1 + self.b1
        self.a1 = self.relu(self.z1)
        
        # Layer 2
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = self.relu(self.z2)
        
        # Output layer
        self.z3 = self.a2 @ self.W3 + self.b3
        self.output = self.softmax(self.z3)
        
        return self.output
    
    def backward(self, y_true, learning_rate=0.001):
        """Backward pass with gradient descent."""
        m = y_true.shape[0]
        
        # One-hot encode labels
        y_onehot = np.zeros((m, self.n_classes))
        y_onehot[np.arange(m), y_true] = 1
        
        # Output layer gradient
        dz3 = self.output - y_onehot
        dW3 = (self.a2.T @ dz3) / m
        db3 = np.mean(dz3, axis=0)
        
        # Layer 2 gradient
        dz2 = (dz3 @ self.W3.T) * self.relu_derivative(self.z2)
        dW2 = (self.a1.T @ dz2) / m
        db2 = np.mean(dz2, axis=0)
        
        # Layer 1 gradient
        dz1 = (dz2 @ self.W2.T) * self.relu_derivative(self.z1)
        dW1 = (self.X_flat.T @ dz1) / m
        db1 = np.mean(dz1, axis=0)
        
        # Update weights
        self.W3 -= learning_rate * dW3
        self.b3 -= learning_rate * db3
        self.W2 -= learning_rate * dW2
        self.b2 -= learning_rate * db2
        self.W1 -= learning_rate * dW1
        self.b1 -= learning_rate * db1
    
    def compute_loss(self, y_true):
        """Cross-entropy loss."""
        m = y_true.shape[0]
        log_probs = -np.log(self.output[np.arange(m), y_true] + 1e-10)
        return np.mean(log_probs)
    
    def predict(self, X):
        """Predict class labels."""
        probs = self.forward(X)
        return np.argmax(probs, axis=1)
    
    def predict_proba(self, X):
        """Predict class probabilities."""
        return self.forward(X)


# Create model
model = SimpleCNN(
    input_shape=X_train.shape[1:],
    n_classes=len(class_names),
    hidden_units=256
)

print(f"Model input shape: {model.input_shape}")
print(f"Flattened input size: {model.input_size}")
print(f"Hidden units: {model.hidden_units}")
print(f"Output classes: {model.n_classes}")

## 5. Training Loop with Monitoring

In [ ]:
def train_model(model, X_train, y_train, X_val, y_val, 
                epochs=50, batch_size=64, learning_rate=0.01):
    """
    Train model with validation monitoring.
    """
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': []
    }
    
    n_samples = X_train.shape[0]
    n_batches = n_samples // batch_size
    
    best_val_acc = 0
    best_weights = None
    
    for epoch in range(epochs):
        # Shuffle training data
        indices = np.random.permutation(n_samples)
        X_shuffled = X_train[indices]
        y_shuffled = y_train[indices]
        
        epoch_loss = 0
        
        # Mini-batch training
        for i in range(n_batches):
            start_idx = i * batch_size
            end_idx = start_idx + batch_size
            
            X_batch = X_shuffled[start_idx:end_idx]
            y_batch = y_shuffled[start_idx:end_idx]
            
            # Forward and backward pass
            model.forward(X_batch)
            batch_loss = model.compute_loss(y_batch)
            model.backward(y_batch, learning_rate)
            
            epoch_loss += batch_loss
        
        # Calculate metrics
        train_loss = epoch_loss / n_batches
        train_pred = model.predict(X_train)
        train_acc = np.mean(train_pred == y_train)
        
        model.forward(X_val)
        val_loss = model.compute_loss(y_val)
        val_pred = model.predict(X_val)
        val_acc = np.mean(val_pred == y_val)
        
        # Record history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = {
                'W1': model.W1.copy(), 'b1': model.b1.copy(),
                'W2': model.W2.copy(), 'b2': model.b2.copy(),
                'W3': model.W3.copy(), 'b3': model.b3.copy()
            }
        
        # Print progress
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:3d}: "
                  f"Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, "
                  f"Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")
    
    # Restore best weights
    if best_weights:
        model.W1, model.b1 = best_weights['W1'], best_weights['b1']
        model.W2, model.b2 = best_weights['W2'], best_weights['b2']
        model.W3, model.b3 = best_weights['W3'], best_weights['b3']
    
    return history


# Train model
print("Training CNN model...\n")
history = train_model(
    model, X_train_aug, y_train_aug, X_val, y_val,
    epochs=50, batch_size=64, learning_rate=0.01
)

print(f"\nTraining complete! Best validation accuracy: {max(history['val_acc']):.4f}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history['train_acc'], label='Train Accuracy')
axes[1].plot(history['val_acc'], label='Val Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 6. Model Evaluation

In [ ]:
# Evaluate on test set
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

test_accuracy = np.mean(y_pred == y_test)
print(f"Test Accuracy: {test_accuracy:.4f}")
print("\n" + "="*50)
print("Classification Report:")
print("="*50)
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
# Confusion matrix
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Per-class accuracy analysis
class_accuracies = []
for i, name in enumerate(class_names):
    mask = y_test == i
    acc = np.mean(y_pred[mask] == y_test[mask])
    class_accuracies.append(acc)
    print(f"{name.title():12} Accuracy: {acc:.4f}")

# Bar chart
plt.figure(figsize=(10, 5))
colors = plt.cm.viridis(np.linspace(0.3, 0.8, len(class_names)))
plt.bar(class_names, class_accuracies, color=colors)
plt.ylabel('Accuracy')
plt.title('Per-Class Accuracy')
plt.ylim(0, 1)
for i, acc in enumerate(class_accuracies):
    plt.text(i, acc + 0.02, f'{acc:.2f}', ha='center')
plt.tight_layout()
plt.show()

## 7. Visualize Predictions

In [ ]:
# Show sample predictions
fig, axes = plt.subplots(3, 5, figsize=(15, 10))

# Get random samples
sample_indices = np.random.choice(len(X_test), 15, replace=False)

for idx, ax in enumerate(axes.flat):
    sample_idx = sample_indices[idx]
    img = X_test[sample_idx]
    true_label = y_test[sample_idx]
    pred_label = y_pred[sample_idx]
    confidence = y_proba[sample_idx, pred_label]
    
    ax.imshow(img)
    
    color = 'green' if true_label == pred_label else 'red'
    ax.set_title(f'True: {class_names[true_label]}\n'
                 f'Pred: {class_names[pred_label]} ({confidence:.2f})',
                 color=color, fontsize=10)
    ax.axis('off')

plt.suptitle('Sample Predictions (Green=Correct, Red=Incorrect)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Show misclassified examples
misclassified = np.where(y_pred != y_test)[0]

if len(misclassified) > 0:
    n_show = min(10, len(misclassified))
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    
    for idx, ax in enumerate(axes.flat):
        if idx < len(misclassified):
            sample_idx = misclassified[idx]
            img = X_test[sample_idx]
            true_label = y_test[sample_idx]
            pred_label = y_pred[sample_idx]
            confidence = y_proba[sample_idx, pred_label]
            
            ax.imshow(img)
            ax.set_title(f'True: {class_names[true_label]}\n'
                         f'Pred: {class_names[pred_label]} ({confidence:.2f})',
                         color='red', fontsize=10)
        ax.axis('off')
    
    plt.suptitle(f'Misclassified Examples ({len(misclassified)} total)', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No misclassified examples!")

## 8. Inference Pipeline

In [ ]:
class ImageClassifier:
    """
    Production-ready image classifier with preprocessing and inference.
    """
    
    def __init__(self, model, class_names, input_size=32):
        self.model = model
        self.class_names = class_names
        self.input_size = input_size
    
    def preprocess(self, image):
        """
        Preprocess image for inference.
        """
        # Ensure image is numpy array
        if not isinstance(image, np.ndarray):
            raise ValueError("Image must be a numpy array")
        
        # Handle single image
        if image.ndim == 3:
            image = image[np.newaxis, ...]
        
        # Normalize to [0, 1]
        if image.max() > 1.0:
            image = image / 255.0
        
        return image
    
    def predict(self, image, top_k=3):
        """
        Predict class for an image with confidence scores.
        """
        # Preprocess
        processed = self.preprocess(image)
        
        # Get predictions
        probs = self.model.predict_proba(processed)[0]
        
        # Get top k predictions
        top_indices = np.argsort(probs)[::-1][:top_k]
        
        results = []
        for idx in top_indices:
            results.append({
                'class': self.class_names[idx],
                'class_id': int(idx),
                'confidence': float(probs[idx])
            })
        
        return {
            'prediction': self.class_names[top_indices[0]],
            'confidence': float(probs[top_indices[0]]),
            'top_predictions': results
        }
    
    def predict_batch(self, images):
        """
        Predict classes for a batch of images.
        """
        processed = self.preprocess(images)
        predictions = self.model.predict(processed)
        probs = self.model.predict_proba(processed)
        
        results = []
        for i, pred in enumerate(predictions):
            results.append({
                'prediction': self.class_names[pred],
                'confidence': float(probs[i, pred])
            })
        
        return results


# Create classifier
classifier = ImageClassifier(model, class_names)

# Test inference
test_image = X_test[0]
result = classifier.predict(test_image)

print("=== Inference Result ===")
print(f"Prediction: {result['prediction']}")
print(f"Confidence: {result['confidence']:.4f}")
print(f"\nTop predictions:")
for pred in result['top_predictions']:
    print(f"  {pred['class']}: {pred['confidence']:.4f}")

In [ ]:
# Batch inference test
batch_results = classifier.predict_batch(X_test[:5])

print("=== Batch Inference Results ===")
for i, res in enumerate(batch_results):
    true_class = class_names[y_test[i]]
    status = '✓' if res['prediction'] == true_class else '✗'
    print(f"Image {i}: {res['prediction']} ({res['confidence']:.3f}) "
          f"[True: {true_class}] {status}")

## 9. Model Serialization

In [ ]:
import joblib

# Save model
model_path = Path('./models')
model_path.mkdir(exist_ok=True)

# Save weights and config
model_data = {
    'weights': {
        'W1': model.W1, 'b1': model.b1,
        'W2': model.W2, 'b2': model.b2,
        'W3': model.W3, 'b3': model.b3
    },
    'config': {
        'input_shape': model.input_shape,
        'n_classes': model.n_classes,
        'hidden_units': model.hidden_units
    },
    'class_names': class_names,
    'metrics': {
        'test_accuracy': test_accuracy,
        'best_val_accuracy': max(history['val_acc'])
    }
}

joblib.dump(model_data, model_path / 'image_classifier.joblib')
print(f"Model saved to {model_path / 'image_classifier.joblib'}")

In [ ]:
# Load and verify model
loaded_data = joblib.load(model_path / 'image_classifier.joblib')

# Reconstruct model
loaded_model = SimpleCNN(
    input_shape=loaded_data['config']['input_shape'],
    n_classes=loaded_data['config']['n_classes'],
    hidden_units=loaded_data['config']['hidden_units']
)

# Load weights
loaded_model.W1 = loaded_data['weights']['W1']
loaded_model.b1 = loaded_data['weights']['b1']
loaded_model.W2 = loaded_data['weights']['W2']
loaded_model.b2 = loaded_data['weights']['b2']
loaded_model.W3 = loaded_data['weights']['W3']
loaded_model.b3 = loaded_data['weights']['b3']

# Verify
loaded_pred = loaded_model.predict(X_test)
loaded_acc = np.mean(loaded_pred == y_test)

print(f"Original model accuracy: {test_accuracy:.4f}")
print(f"Loaded model accuracy: {loaded_acc:.4f}")
print(f"Match: {np.allclose(y_pred, loaded_pred)}")

## Summary

### Skills Demonstrated

- **Data augmentation**: Custom transformations for image data
- **CNN architecture**: Multi-layer neural network for image classification
- **Training pipeline**: Mini-batch training with validation monitoring
- **Model evaluation**: Confusion matrix, per-class metrics
- **Inference pipeline**: Production-ready classifier class
- **Model serialization**: Save/load model for deployment

### Key Takeaways

1. Data augmentation increases effective training data
2. Monitor validation metrics to prevent overfitting
3. Save best model during training
4. Create clean inference API for production use
5. Visualize predictions to understand model behavior